# CUDA Optimization Course - Module 4: Gradient Checkpointing

Trade memory for compute to enable training of larger models.

## Setup

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.checkpoint import checkpoint, checkpoint_sequential
import time

if torch.cuda.is_available():
    device = 'cuda'
elif torch.backends.mps.is_available():
    device = 'mps'
else:
    device = 'cpu'

print(f"Device: {device}")

Device: mps


## Understanding Gradient Checkpointing

Normally, during backpropagation, we need to store activations from the forward pass.
This uses significant memory for large models.

**Gradient Checkpointing** recomputes activations during backward pass instead of storing them.
- **Memory**: Reduced (typically 50% savings)
- **Computation**: Increased (recomputation during backprop)
- **Best for**: Large models where memory is the bottleneck

## Large Transformer Model

In [2]:
class AttentionBlock(nn.Module):
    def __init__(self, d_model=512, nhead=8):
        super().__init__()
        self.attention = nn.MultiheadAttention(d_model, nhead, batch_first=True)
        self.norm1 = nn.LayerNorm(d_model)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, 2048),
            nn.ReLU(),
            nn.Linear(2048, d_model)
        )
        self.norm2 = nn.LayerNorm(d_model)
    
    def forward(self, x):
        # Self-attention with residual
        attn_out, _ = self.attention(x, x, x)
        x = x + attn_out
        x = self.norm1(x)
        
        # FFN with residual
        ffn_out = self.ffn(x)
        x = x + ffn_out
        x = self.norm2(x)
        
        return x

class LargeTransformer(nn.Module):
    def __init__(self, vocab_size=10000, d_model=512, nhead=8, num_layers=12):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.layers = nn.ModuleList([
            AttentionBlock(d_model, nhead) for _ in range(num_layers)
        ])
        self.fc = nn.Linear(d_model, vocab_size)
    
    def forward(self, x):
        x = self.embedding(x)
        for layer in self.layers:
            x = layer(x)
        x = self.fc(x)
        return x

model = LargeTransformer(num_layers=12).to(device)
print(f"Model parameters: {sum(p.numel() for p in model.parameters()) / 1e6:.1f}M")

Model parameters: 48.1M


## Baseline: Without Gradient Checkpointing

In [3]:
batch_size = 16
seq_length = 256

input_ids = torch.randint(0, 10000, (batch_size, seq_length)).to(device)
target_ids = torch.randint(0, 10000, (batch_size, seq_length)).to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)

# Warmup
optimizer.zero_grad()
output = model(input_ids)
loss = F.cross_entropy(output.view(-1, 10000), target_ids.view(-1))
loss.backward()
optimizer.step()

# Measure without checkpointing
torch.mps.synchronize() if device == 'mps' else torch.cuda.synchronize() if device == 'cuda' else None
start = time.perf_counter()

num_iterations = 10
for _ in range(num_iterations):
    optimizer.zero_grad()
    output = model(input_ids)
    loss = F.cross_entropy(output.view(-1, 10000), target_ids.view(-1))
    loss.backward()
    optimizer.step()

torch.mps.synchronize() if device == 'mps' else torch.cuda.synchronize() if device == 'cuda' else None
baseline_time = time.perf_counter() - start

print(f"Without Gradient Checkpointing:")
print(f"Time for {num_iterations} iterations: {baseline_time:.2f}s")
print(f"Time per iteration: {baseline_time / num_iterations * 1000:.2f}ms")

Without Gradient Checkpointing:
Time for 10 iterations: 2.22s
Time per iteration: 221.87ms


## Model with Gradient Checkpointing

In [4]:
class CheckpointedTransformer(nn.Module):
    def __init__(self, vocab_size=10000, d_model=512, nhead=8, num_layers=12):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.layers = nn.ModuleList([
            AttentionBlock(d_model, nhead) for _ in range(num_layers)
        ])
        self.fc = nn.Linear(d_model, vocab_size)
    
    def forward(self, x):
        x = self.embedding(x)
        
        for i, layer in enumerate(self.layers):
            # Wrap each layer with checkpoint
            # During backprop, layer input is recomputed instead of stored
            x = checkpoint(layer, x, use_reentrant=False)
        
        x = self.fc(x)
        return x

model_ckpt = CheckpointedTransformer(num_layers=12).to(device)
# Copy weights from baseline model
model_ckpt.load_state_dict(model.state_dict())

optimizer_ckpt = torch.optim.AdamW(model_ckpt.parameters(), lr=1e-4)

# Warmup
optimizer_ckpt.zero_grad()
output = model_ckpt(input_ids)
loss = F.cross_entropy(output.view(-1, 10000), target_ids.view(-1))
loss.backward()
optimizer_ckpt.step()

# Measure with checkpointing
torch.mps.synchronize() if device == 'mps' else torch.cuda.synchronize() if device == 'cuda' else None
start = time.perf_counter()

for _ in range(num_iterations):
    optimizer_ckpt.zero_grad()
    output = model_ckpt(input_ids)
    loss = F.cross_entropy(output.view(-1, 10000), target_ids.view(-1))
    loss.backward()
    optimizer_ckpt.step()

torch.mps.synchronize() if device == 'mps' else torch.cuda.synchronize() if device == 'cuda' else None
ckpt_time = time.perf_counter() - start

print(f"With Gradient Checkpointing:")
print(f"Time for {num_iterations} iterations: {ckpt_time:.2f}s")
print(f"Time per iteration: {ckpt_time / num_iterations * 1000:.2f}ms")
print(f"\nSlowdown due to recomputation: {ckpt_time / baseline_time:.2f}x")
print(f"Memory savings: ~50% (typical)")

With Gradient Checkpointing:
Time for 10 iterations: 2.18s
Time per iteration: 217.56ms

Slowdown due to recomputation: 0.98x
Memory savings: ~50% (typical)


## Selective Gradient Checkpointing

Only checkpoint expensive layers (attention) to balance memory/speed.

In [5]:
class SelectiveCheckpointTransformer(nn.Module):
    def __init__(self, vocab_size=10000, d_model=512, nhead=8, num_layers=12):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.layers = nn.ModuleList([
            AttentionBlock(d_model, nhead) for _ in range(num_layers)
        ])
        self.fc = nn.Linear(d_model, vocab_size)
    
    def forward(self, x):
        x = self.embedding(x)
        
        for i, layer in enumerate(self.layers):
            # Checkpoint only even-indexed layers
            if i % 2 == 0:
                x = checkpoint(layer, x, use_reentrant=False)
            else:
                x = layer(x)
        
        x = self.fc(x)
        return x

model_selective = SelectiveCheckpointTransformer(num_layers=12).to(device)
model_selective.load_state_dict(model.state_dict())

optimizer_selective = torch.optim.AdamW(model_selective.parameters(), lr=1e-4)

# Warmup
optimizer_selective.zero_grad()
output = model_selective(input_ids)
loss = F.cross_entropy(output.view(-1, 10000), target_ids.view(-1))
loss.backward()
optimizer_selective.step()

# Measure
torch.mps.synchronize() if device == 'mps' else torch.cuda.synchronize() if device == 'cuda' else None
start = time.perf_counter()

for _ in range(num_iterations):
    optimizer_selective.zero_grad()
    output = model_selective(input_ids)
    loss = F.cross_entropy(output.view(-1, 10000), target_ids.view(-1))
    loss.backward()
    optimizer_selective.step()

torch.mps.synchronize() if device == 'mps' else torch.cuda.synchronize() if device == 'cuda' else None
selective_time = time.perf_counter() - start

print(f"Selective Gradient Checkpointing (even layers only):")
print(f"Time for {num_iterations} iterations: {selective_time:.2f}s")
print(f"Time per iteration: {selective_time / num_iterations * 1000:.2f}ms")
print(f"\nComparison:")
print(f"Baseline:     {baseline_time:.2f}s (1.0x)")
print(f"Selective:    {selective_time:.2f}s ({selective_time/baseline_time:.2f}x)")
print(f"Full ckpt:    {ckpt_time:.2f}s ({ckpt_time/baseline_time:.2f}x)")

Selective Gradient Checkpointing (even layers only):
Time for 10 iterations: 2.02s
Time per iteration: 202.24ms

Comparison:
Baseline:     2.22s (1.0x)
Selective:    2.02s (0.91x)
Full ckpt:    2.18s (0.98x)


## Checkpoint Best Practices

```python
# 1. Basic checkpointing
output = checkpoint(module, input, use_reentrant=False)

# 2. Checkpointing a sequence of modules
x = checkpoint_sequential(modules, segments, input)

# 3. Selective checkpointing (balance memory/speed)
if i > num_layers // 2:  # Only checkpoint later layers
    x = checkpoint(layer, x)

# 4. With key-value cache (important for LLMs)
output = checkpoint(layer, x, cache, use_reentrant=False)
```

## Key Takeaways

1. **Memory Savings**: ~50% reduction in peak memory
2. **Trade-off**: Slower training due to recomputation (~20-30% slower)
3. **Use Case**: Large models where memory is bottleneck
4. **Selective**: Only checkpoint expensive layers for better speed
5. **LLMs**: Essential for training large transformers
6. **use_reentrant=False**: More efficient in modern PyTorch versions